# 🇮🇳 Citizen AI — QLoRA Fine-Tuning on Google Colab (Free T4 GPU)

**Project:** Citizen AI (Indian Government Schemes & Welfare Assistant)  
**Base Model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Method:** 4-Bit QLoRA (Quantized Low-Rank Adaptation) via `PEFT` + `TRL` + `bitsandbytes`  
**Target Hardware:** Free Google Colab Tesla T4 GPU (~2.8 GB VRAM used out of 15.0 GB available)  
**Expected Training Time:** ~8–12 minutes for 3 epochs on 900 training samples

--- 
## 1. Verify GPU Runtime & Install Dependencies
Make sure your Colab Runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi

# Install lightweight fine-tuning stack
!pip install -q --upgrade pip
!pip install -q transformers>=4.45.0 datasets>=3.0.0 accelerate>=0.34.0 peft>=0.13.0 trl>=0.11.0 bitsandbytes>=0.44.0

--- 
## 2. Load & Verify Training / Validation Datasets

In [ ]:
import json
from datasets import load_dataset

# Load local jsonl files (ensure train.jsonl and val.jsonl are in current directory)
dataset = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})
print(dataset)

print("\n--- Sample Training Item ---")
sample_msgs = dataset["train"][0]["messages"]
for msg in sample_msgs:
    print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")

--- 
## 3. Load Qwen2.5-1.5B-Instruct in 4-bit Quantization

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-Bit NormalFloat Quantization (BitsAndBytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"✅ Successfully loaded {MODEL_ID} in 4-bit mode on GPU!")

--- 
## 4. Configure QLoRA Adapters via PEFT

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# Target all linear attention and MLP projection layers for maximum domain grounding
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

--- 
## 5. Train with HuggingFace TRL (SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./citizen-ai-qwen-1.5b-output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=1024,
    dataset_text_field="text",
    report_to="none"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args
)

print("🚀 Starting QLoRA Fine-Tuning...")
trainer.train()
print("✅ Training Complete!")

--- 
## 6. Save LoRA Adapters

In [ ]:
OUTPUT_ADAPTER_DIR = "./citizen-ai-qwen-1.5b-lora"
trainer.model.save_pretrained(OUTPUT_ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)
print(f"💾 LoRA Adapters saved to: {OUTPUT_ADAPTER_DIR}")

# Zip adapter weights for easy download to local machine
!zip -r citizen_ai_lora_weights.zip ./citizen-ai-qwen-1.5b-lora

--- 
## 7. Inference & Qualitative Benchmark with Citations

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=peft_model, tokenizer=tokenizer)

test_queries = [
    "What are the 18 traditional trades covered under PM Vishwakarma?",
    "How much annual financial benefit is given under PM-KISAN?",
    "Compare the loan facilities between PM SVANidhi and PM Vishwakarma.",
    "How do I bake a chocolate cake at home?"
]

for q in test_queries:
    messages = [
        {"role": "system", "content": "You are Citizen AI, a trusted, citation-grounded assistant for Indian Government schemes and welfare policies."},
        {"role": "user", "content": q}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=256, temperature=0.1, do_sample=False)
    generated_text = out[0]["generated_text"][len(prompt):]
    
    print("=" * 70)
    print(f"❓ Query: {q}")
    print("💡 Response:")
    print(generated_text.strip())
    print("=" * 70)